In [4]:
# Step 1: Load Data and Model Pipelines

import pandas as pd
import joblib


In [6]:
# 1. Load Data (step out of 'notebooks/' using '..')
df = pd.read_csv("../data/WA_Fn-UseC_-Accounts-Receivable.csv")


In [7]:
# 2. Load Pre-trained Models
classifier = joblib.load("../models/stage_1_classifier.pkl")
regressor = joblib.load("../models/stage_2_regressor.pkl")

In [9]:
# Convert date columns to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['DueDate'] = pd.to_datetime(df['DueDate'])
df['PaperlessDate'] = pd.to_datetime(df['PaperlessDate'])

# 1. Feature Engineering
df['invoice_month'] = df['InvoiceDate'].dt.month
df['invoice_dayofweek'] = df['InvoiceDate'].dt.dayofweek
df['payment_terms_days'] = (df['DueDate'] - df['InvoiceDate']).dt.days
df['days_since_paperless'] = (df['InvoiceDate'] - df['PaperlessDate']).dt.days

# Fill any NaN values if present
df['days_since_paperless'] = df['days_since_paperless'].fillna(0)
df['payment_terms_days'] = df['payment_terms_days'].fillna(0)

In [11]:
# Stage 1: Predict delayed status
df['is_delayed'] = classifier.predict(df)

# Filter delayed records
delayed_invoices = df[df['is_delayed'] == 1].copy()

print(f"Total: {len(df)} | Delayed: {len(delayed_invoices)}")

Total: 2466 | Delayed: 761


In [13]:
import smtplib
from email.mime.text import MIMEText

# 1. Grab the first flagged invoice from Stage 1
sample = delayed_invoices.iloc[0]

# 2. Email Details
sender = "sajesh.nair.ai@gmail.com"
app_password = "ypqb bhys jkoq vcqg"
receiver = "sajesh.nair.ai@gmail.com"

# 3. Message Body
body = f"""Hello,

This is a reminder regarding Invoice #{sample['invoiceNumber']} for ${sample['InvoiceAmount']:,.2f}.

Our system flagged this account for payment follow-up. Please let us know if you need any assistance or updated payment details.

Best regards,
AutoCollect AI Team"""

msg = MIMEText(body)
msg['Subject'] = f"Payment Reminder: Invoice #{sample['invoiceNumber']}"
msg['From'] = sender
msg['To'] = receiver

# 4. Send Email
try:
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
        server.login(sender, app_password)
        server.send_message(msg)
    print("✅ Email sent successfully!")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Email sent successfully!
